In [ ]:
import pandas as pd
import numpy as np
import math

# =============================================================================
# Mobile Plan Analysis - 36 months (Jan 2016 - Dec 2018)
# =============================================================================

# Plan details from the introduction table
plans = {
    'Run':   {'cost': 60, 'inc_min': 400,        'inc_text': float('inf'), 'text_carry': False, 'inc_data': 1250,        'extra_min_rate': 0.10, 'extra_text_rate': 0,    'data_pack_cost': 10, 'data_pack_mb': 50},
    'Swim':  {'cost': 80, 'inc_min': 180,        'inc_text': 1200,        'text_carry': True,  'inc_data': float('inf'), 'extra_min_rate': 0.69, 'extra_text_rate': 0.04, 'data_pack_cost': 0,  'data_pack_mb': 0},
    'Bike':  {'cost': 80, 'inc_min': float('inf'), 'inc_text': 750,       'text_carry': False, 'inc_data': 2000,        'extra_min_rate': 0,    'extra_text_rate': 0.10, 'data_pack_cost': 35, 'data_pack_mb': 100},
    'Jump':  {'cost': 75, 'inc_min': float('inf'), 'inc_text': 500,       'text_carry': True,  'inc_data': 1500,        'extra_min_rate': 0,    'extra_text_rate': 0.05, 'data_pack_cost': 20, 'data_pack_mb': 200},
    'Kayak': {'cost': 60, 'inc_min': 320,        'inc_text': float('inf'), 'text_carry': False, 'inc_data': 3000,        'extra_min_rate': 0.99, 'extra_text_rate': 0,    'data_pack_cost': 50, 'data_pack_mb': 50},
    'Skip':  {'cost': 10, 'inc_min': 150,        'inc_text': 3000,        'text_carry': False, 'inc_data': 1250,        'extra_min_rate': 0.59, 'extra_text_rate': 0.06, 'data_pack_cost': 45, 'data_pack_mb': 300},
}

# Usage parameters
base_minutes = 300
base_texts = 1000
base_data = 2000  # MB

# Build monthly usage for 36 months
# Jan 2016 = month_index 1, Dec 2018 = month_index 36
months = []
for year_offset in range(3):  # 2016, 2017, 2018
    year = 2016 + year_offset
    annual_factor = 1.05 ** year_offset
    for month_num in range(1, 13):
        if month_num in [6, 7]:
            seasonal_factor = 1.5
        elif month_num == 12:
            seasonal_factor = 2.0
        else:
            seasonal_factor = 1.0
        
        usage_min = base_minutes * annual_factor * seasonal_factor
        usage_text = base_texts * annual_factor * seasonal_factor
        usage_data = base_data * annual_factor * seasonal_factor
        
        months.append({
            'month_index': len(months) + 1,
            'year': year,
            'month_num': month_num,
            'usage_min': usage_min,
            'usage_text': usage_text,
            'usage_data': usage_data,
        })

print(f"Total months: {len(months)}")
for i in [0, 5, 11, 12, 35]:
    m = months[i]
    print(f"  Month {m['month_index']} ({m['year']}-{m['month_num']:02d}): min={m['usage_min']:.2f}, text={m['usage_text']:.2f}, data={m['usage_data']:.2f}")

In [ ]:
# =============================================================================
# Simulate each plan over 36 months (NO incentives)
# Questions 1-6 use no incentives
# =============================================================================

def simulate_plan(plan_name, plan, months_list, use_incentive=False, usage_multiplier=1.0):
    """
    Simulate a mobile plan over the given months.
    Returns a list of dicts with monthly cost breakdowns.
    """
    results = []
    carried_minutes = 0
    carried_texts = 0
    
    for m in months_list:
        mi = m['month_index']  # 1-based month index
        usage_min = m['usage_min'] * usage_multiplier
        usage_text = m['usage_text'] * usage_multiplier
        usage_data = m['usage_data'] * usage_multiplier
        
        # --- Determine effective included amounts (with incentives if applicable) ---
        inc_min = plan['inc_min']
        inc_text = plan['inc_text']
        inc_data = plan['inc_data']
        base_cost = plan['cost']
        
        if use_incentive:
            if plan_name == 'Kayak' and mi <= 4:
                inc_min = inc_min * 2  # Double included minutes for first 4 months
            if plan_name == 'Jump' and mi <= 9:
                inc_data = inc_data + 1000  # Extra 1000MB for first 9 months
            if plan_name == 'Skip' and mi <= 3:
                inc_data = inc_data * 2  # Double included data for first 3 months
        
        # --- Minutes ---
        available_min = inc_min + carried_minutes if inc_min != float('inf') else float('inf')
        if available_min == float('inf'):
            extra_min = 0
            carried_minutes = 0  # No carry for unlimited
        elif usage_min <= available_min:
            extra_min = 0
            carried_minutes = available_min - usage_min  # Unused minutes carry forward
        else:
            extra_min = usage_min - available_min
            carried_minutes = 0
        
        extra_min_cost = extra_min * plan['extra_min_rate']
        
        # --- Text messages ---
        if plan['text_carry']:
            available_text = inc_text + carried_texts
        else:
            available_text = inc_text
            carried_texts = 0  # Reset carried texts if no carry forward
        
        if available_text == float('inf'):
            extra_text = 0
            carried_texts = 0
        elif usage_text <= available_text:
            extra_text = 0
            if plan['text_carry']:
                carried_texts = available_text - usage_text
            else:
                carried_texts = 0
        else:
            extra_text = usage_text - available_text
            carried_texts = 0
        
        extra_text_cost = extra_text * plan['extra_text_rate']
        
        # --- Data ---
        # Data does NOT carry forward
        if inc_data == float('inf'):
            extra_data_packs = 0
        elif usage_data <= inc_data:
            extra_data_packs = 0
        else:
            extra_data_needed = usage_data - inc_data
            if plan['data_pack_mb'] > 0:
                extra_data_packs = math.ceil(extra_data_needed / plan['data_pack_mb'])
            else:
                extra_data_packs = 0
        
        extra_data_cost = extra_data_packs * plan['data_pack_cost']
        
        # --- Total monthly cost ---
        additional_usage_cost = extra_min_cost + extra_text_cost + extra_data_cost
        total_before_incentive = base_cost + additional_usage_cost
        
        # Apply incentive discounts to cost
        discount = 0
        if use_incentive:
            if plan_name == 'Run' and mi <= 2:
                discount = base_cost  # First two months base cost free
            elif plan_name == 'Swim' and mi <= 6:
                discount = base_cost * 0.5  # 50% discount on base cost
            elif plan_name == 'Bike' and mi <= 3:
                discount = total_before_incentive * 0.6  # 60% discount on total cost
        
        total_monthly = total_before_incentive - discount
        
        results.append({
            'month_index': mi,
            'year': m['year'],
            'month_num': m['month_num'],
            'base_cost': base_cost,
            'usage_min': usage_min,
            'usage_text': usage_text,
            'usage_data': usage_data,
            'inc_min': inc_min,
            'inc_text': inc_text if inc_text != float('inf') else 'Unlimited',
            'inc_data': inc_data,
            'extra_min': extra_min,
            'extra_min_cost': extra_min_cost,
            'extra_text': extra_text,
            'extra_text_cost': extra_text_cost,
            'extra_data_packs': extra_data_packs,
            'extra_data_cost': extra_data_cost,
            'additional_usage_cost': additional_usage_cost,
            'discount': discount,
            'total_monthly': total_monthly,
        })
    
    return results

# Run simulation for all plans without incentives
results_no_incentive = {}
for name, plan in plans.items():
    results_no_incentive[name] = simulate_plan(name, plan, months, use_incentive=False)

# Run simulation for all plans with incentives
results_with_incentive = {}
for name, plan in plans.items():
    results_with_incentive[name] = simulate_plan(name, plan, months, use_incentive=True)

print("Simulations complete (with and without incentives).")

In [ ]:
# =============================================================================
# Answer Questions 1-6 (no incentives)
# =============================================================================

# Q1: Kayak plan, total cost for additional calls for 2018
kayak_2018 = [r for r in results_no_incentive['Kayak'] if r['year'] == 2018]
q1_val = sum(r['extra_min_cost'] for r in kayak_2018)
print(f"Q1: Kayak additional call cost 2018 = ${q1_val:.2f}")
# Map to multiple choice
q1_options = {'A': 782.50, 'B': 782.60, 'C': 782.70, 'D': 782.80, 'E': 782.90, 'F': 783.00}
q1_answer = min(q1_options.keys(), key=lambda k: abs(q1_options[k] - q1_val))
print(f"Q1 answer: {q1_answer}")

# Q2: Swim plan, total cost for additional text messages in December 2018
swim_results = results_no_incentive['Swim']
swim_dec2018 = [r for r in swim_results if r['year'] == 2018 and r['month_num'] == 12]
q2_val = swim_dec2018[0]['extra_text_cost']
print(f"\nQ2: Swim additional text cost Dec 2018 = ${q2_val:.4f}")
q2_options = {'A': 24.52, 'B': 24.56, 'C': 24.60, 'D': 24.64, 'E': 24.68, 'F': 24.72}
q2_answer = min(q2_options.keys(), key=lambda k: abs(q2_options[k] - q2_val))
print(f"Q2 answer: {q2_answer}")

# Q3: Skip plan, total cost of additional data over 36 months
skip_results = results_no_incentive['Skip']
q3_val = sum(r['extra_data_cost'] for r in skip_results)
print(f"\nQ3: Skip additional data cost 36 months = ${q3_val:.2f}")
q3_options = {'A': 7110, 'B': 7155, 'C': 7200, 'D': 7245, 'E': 7290, 'F': 7335}
q3_answer = min(q3_options.keys(), key=lambda k: abs(q3_options[k] - q3_val))
print(f"Q3 answer: {q3_answer}")

# Q4: Bike plan, cumulative total cost up to and including October 2018
# Oct 2018 = month_index 34 (month 1 = Jan 2016, month 34 = Oct 2018)
bike_results = results_no_incentive['Bike']
bike_up_to_oct2018 = [r for r in bike_results if r['month_index'] <= 34]
q4_val = sum(r['total_monthly'] for r in bike_up_to_oct2018)
print(f"\nQ4: Bike cumulative cost up to Oct 2018 = ${q4_val:.2f}")
q4_options = {'A': 9397.55, 'B': 9397.65, 'C': 9397.75, 'D': 9397.85, 'E': 9397.95, 'F': 9398.05}
q4_answer = min(q4_options.keys(), key=lambda k: abs(q4_options[k] - q4_val))
print(f"Q4 answer: {q4_answer}")

# Q5: Cheapest plan over 36 months (no incentives)
total_costs_no_incentive = {}
for name in plans:
    total_costs_no_incentive[name] = sum(r['total_monthly'] for r in results_no_incentive[name])
    print(f"\n{name} total 36-month cost (no incentive): ${total_costs_no_incentive[name]:.2f}")

cheapest_no_incentive = min(total_costs_no_incentive, key=total_costs_no_incentive.get)
print(f"\nQ5: Cheapest plan = {cheapest_no_incentive}")
q5_map = {'Run': 'A', 'Swim': 'B', 'Bike': 'C', 'Jump': 'D', 'Kayak': 'E', 'Skip': 'F'}
q5_answer = q5_map[cheapest_no_incentive]
print(f"Q5 answer: {q5_answer}")

# Q6: Length of contract (from Jan 2016) where Bike and Jump have the same total cost
# Compare cumulative costs month by month
q6_answer_val = None
for n in range(1, 37):
    bike_cum = sum(r['total_monthly'] for r in results_no_incentive['Bike'][:n])
    jump_cum = sum(r['total_monthly'] for r in results_no_incentive['Jump'][:n])
    if abs(bike_cum - jump_cum) < 0.005:  # Within half a cent
        print(f"\nQ6: Bike and Jump same cost at {n} months (Bike=${bike_cum:.2f}, Jump=${jump_cum:.2f})")
        q6_answer_val = n
        break

if q6_answer_val is None:
    # If exact match not found, check for crossover
    for n in range(1, 37):
        bike_cum = sum(r['total_monthly'] for r in results_no_incentive['Bike'][:n])
        jump_cum = sum(r['total_monthly'] for r in results_no_incentive['Jump'][:n])
        print(f"  Month {n}: Bike=${bike_cum:.2f}, Jump=${jump_cum:.2f}, diff=${bike_cum-jump_cum:.2f}")

q6_options = {'A': 5, 'B': 6, 'C': 7, 'D': 8, 'E': 9, 'F': 10}
if q6_answer_val is not None:
    q6_answer = min(q6_options.keys(), key=lambda k: abs(q6_options[k] - q6_answer_val))
else:
    q6_answer = 'F'  # Default
print(f"Q6 answer: {q6_answer}")

In [ ]:
# =============================================================================
# Answer Questions 7-10 (with incentives)
# =============================================================================

# Q7: Which carrier's incentive gives the LEAST dollar cost saving over 36 months?
savings = {}
for name in plans:
    cost_no = sum(r['total_monthly'] for r in results_no_incentive[name])
    cost_with = sum(r['total_monthly'] for r in results_with_incentive[name])
    saving = cost_no - cost_with
    savings[name] = saving
    print(f"{name}: No incentive=${cost_no:.2f}, With incentive=${cost_with:.2f}, Saving=${saving:.2f}")

least_saving = min(savings, key=savings.get)
print(f"\nQ7: Least saving = {least_saving}")
q7_answer = q5_map[least_saving]
print(f"Q7 answer: {q7_answer}")

# Q8: Cheapest plan with incentives over 36 months
total_costs_with_incentive = {}
for name in plans:
    total_costs_with_incentive[name] = sum(r['total_monthly'] for r in results_with_incentive[name])
    print(f"\n{name} total 36-month cost (with incentive): ${total_costs_with_incentive[name]:.2f}")

cheapest_with_incentive = min(total_costs_with_incentive, key=total_costs_with_incentive.get)
print(f"\nQ8: Cheapest plan with incentive = {cheapest_with_incentive}")
q8_answer = q5_map[cheapest_with_incentive]
print(f"Q8 answer: {q8_answer}")

# Q9: Total cost of all six plans with incentives over 36 months
q9_val = sum(total_costs_with_incentive.values())
print(f"\nQ9: Total cost of all plans with incentives = ${q9_val:.2f}")
q9_options = {'A': 54850.19, 'B': 54850.23, 'C': 54850.27, 'D': 54850.31, 'E': 54850.35, 'F': 54850.39}
q9_answer = min(q9_options.keys(), key=lambda k: abs(q9_options[k] - q9_val))
print(f"Q9 answer: {q9_answer}")

# Q10: Friend with half usage, cheapest with incentives over 36 months
results_friend_incentive = {}
for name, plan in plans.items():
    results_friend_incentive[name] = simulate_plan(name, plan, months, use_incentive=True, usage_multiplier=0.5)

total_costs_friend = {}
for name in plans:
    total_costs_friend[name] = sum(r['total_monthly'] for r in results_friend_incentive[name])
    print(f"\n{name} friend 36-month cost (with incentive): ${total_costs_friend[name]:.2f}")

cheapest_friend = min(total_costs_friend, key=total_costs_friend.get)
print(f"\nQ10: Cheapest for friend = {cheapest_friend}")
q10_answer = q5_map[cheapest_friend]
print(f"Q10 answer: {q10_answer}")

In [ ]:
# =============================================================================
# Set final answers dict
# =============================================================================

answers = {
    'question1': q1_answer,
    'question2': q2_answer,
    'question3': q3_answer,
    'question4': q4_answer,
    'question5': q5_answer,
    'question6': q6_answer,
    'question7': q7_answer,
    'question8': q8_answer,
    'question9': q9_answer,
    'question10': q10_answer,
}

print("\n=== FINAL ANSWERS ===")
for k, v in answers.items():
    print(f"  {k}: {v}")